# Production RAG capstone

**Scenario:** NovaTech is building an Enterprise Knowledge Assistant for HR, finance, IT, project, and vendor knowledge.

Design and test a production-style RAG system: offline pipeline, online answer path, access filters, citations, traces, latency/cost, and abstention.

```mermaid
flowchart LR
  Q[User question] --> R[Retrieve evidence]
  R --> V[Verify / rerank]
  V --> A[Answer with citations or abstain]
  A --> E[Evaluate failure]
```

## Learning pattern

1. Start from a deliberately simple baseline.
2. Break it with a realistic failure case.
3. Improve one component.
4. Measure the change and write down the architecture lesson.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT))

In [ ]:
from src.enterprise_rag.corpus import load_corpus, chunk_documents
from src.enterprise_rag.retrieval import hybrid_retrieve
from src.enterprise_rag.generation import answer_with_citations
from src.enterprise_rag.evaluation import cost_per_successful_task
chunks=chunk_documents(load_corpus(ROOT/'data/enterprise'))
questions=['What increased by 14% in Q2 2025?','What does AX-774-B mean?','Who supplies Project Atlas technology and what regulation applies?']
trace=[]
for q in questions:
    hits=hybrid_retrieve(q,chunks,5)
    ans=answer_with_citations(q,hits)
    trace.append({'question':q,'tool_calls':1,'citations':len(ans['citations']),'supported':ans['supported'],'estimated_cost':0.002})
print(json.dumps(trace, indent=2))
print('cost_per_successful_task', cost_per_successful_task(sum(t['estimated_cost'] for t in trace), sum(t['supported'] for t in trace)))

## Exercise

- Change one query or corpus document.
- Predict which component should fail before you run it.
- Record the retrieval trace, citations, and whether the answer should abstain.

## References

- Lewis et al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks: https://arxiv.org/abs/2005.11401
- Stanford Introduction to Information Retrieval: https://nlp.stanford.edu/IR-book/
- Ragas metrics: https://docs.ragas.io/en/stable/concepts/metrics/
- LangChain retrieval concepts: https://docs.langchain.com/oss/python/langchain/retrieval
- LlamaIndex RAG guide: https://docs.llamaindex.ai/en/stable/understanding/rag/
- Haystack pipelines: https://docs.haystack.deepset.ai/docs/pipelines